In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Values from the channel sheet (lvai.kicad_sch); refs are channel 0's
R2 = 22e3   # input series resistor (terminal -> in_div)
R4 = 22e3   # in_div -> GND; R2/R4 halve the input
R1 = 1e3    # offset buffer (UxA) feedback, follower
R3 = 1e3    # difference stage (UxB): Vref -> UxB-
R5 = 1e3    # difference stage (UxB): UxB- -> UxB out
R6 = 1e3    # gain stage (UxC): wiper -> UxC- (carries no current)
R7 = 1e3    # filter resistor
C6 = 1e-6   # filter capacitor, C0G
RV1 = 10e3  # offset trimmer, 5V -> GND
RV2 = 10e3  # gain trimmer, UxC out -> GND
VDD = 5.0   # op-amp and trimmer supply

# Frequency range for analysis
frequencies = np.logspace(0, 6, 1000)  # 1 Hz to 1 MHz

In [ ]:

def input_divider(Vin):
    """R2/R4 divider into UxB+ (in_div); BAS416 D(5V) holds it near 5.75 V at +24 V; at -24 V the MCP6004's own input diode takes most of the current."""
    return Vin * R4 / (R2 + R4)

def voltage_reference_buffer(RV1_position):
    """Offset trimmer wiper, buffered by UxA (follower): Vref = VDD * R(wiper to GND) / RV1."""
    return VDD * RV1_position / RV1

def differential_amplifier(Vdiv, Vref):
    """UxB: Vout = Vdiv * (1 + R5/R3) - Vref * R5/R3; with R5 = R3 and Vdiv = Vin/2 this is Vin - Vref."""
    return Vdiv * (1 + R5 / R3) - Vref * R5 / R3

def adjustable_gain_amplifier(Vin, RV2_position):
    """UxC, non-inverting: the wiper (through R6, no current) sets UxC-, so G = RV2 / R(wiper to GND)."""
    gain = RV2 / RV2_position
    return gain * Vin, gain

def low_pass_filter_gain(freq):
    """R7/C6 low-pass into the UxD follower."""
    fc = 1 / (2 * np.pi * R7 * C6)
    return 1 / np.sqrt(1 + (freq / fc) ** 2)

def clip(v):
    """Rail-to-rail output on a 0-5 V supply."""
    return min(max(v, 0.0), VDD)

# Parameter Inputs

In [ ]:
# Input on the terminal, 0-10 V
Vin = 4.0

# Trimmer settings: R(wiper to GND) as a fraction of 10k
RV1_ratio = 0.5   # offset: Vref = 2.5 V
RV2_ratio = 0.5   # gain: G = 2

## Intermediate Calculations

In [ ]:

RV1_position = RV1_ratio * RV1
RV2_position = RV2_ratio * RV2

Vdiv = input_divider(Vin)
Vref = voltage_reference_buffer(RV1_position)
Vdiff = clip(differential_amplifier(Vdiv, Vref))
Vgain, gain_stage3 = adjustable_gain_amplifier(Vdiff, RV2_position)
Vgain = clip(Vgain)

gains_stage4 = np.array([low_pass_filter_gain(f) for f in frequencies])
Vout = Vgain * gains_stage4

# Small-signal gain from the terminal to the Nano pin: 0.5 (divider) * 2 (difference) * G * filter
cumulative_gain_decimal = 0.5 * (1 + R5 / R3) * gain_stage3 * gains_stage4
cumulative_gain_db = 20 * np.log10(abs(cumulative_gain_decimal))

cutoff_frequency = 1 / (2 * np.pi * R7 * C6)
max_gain_decimal = np.max(cumulative_gain_decimal)
max_gain_db = np.max(cumulative_gain_db)

# Results

In [ ]:

print(f"Offset (Vref, RV1): {Vref:.2f} V")
print(f"Input: {Vin:.2f} V -> in_div {Vdiv:.2f} V -> difference stage {Vdiff:.2f} V")
print(f"Gain stage G = {gain_stage3:.2f} -> {Vgain:.2f} V")
print(f"Output at 1 Hz: {Vout[0]:.2f} V")

plt.figure(figsize=(10, 6))
plt.semilogx(frequencies, cumulative_gain_db, label="Terminal to Nano pin (dB)")
plt.axvline(cutoff_frequency, color='r', linestyle='--', label=f'Cutoff Frequency = {cutoff_frequency:.2f} Hz')
plt.axhline(max_gain_db, color='g', linestyle='--', label=f'Max Gain = {max_gain_db:.2f} dB ({max_gain_decimal:.2f} decimal)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Gain (dB)')
plt.title('Gain vs Frequency, terminal to Nano pin')
plt.legend()
plt.grid()
plt.show()